# 02 — Data Quality Review

**Goal:** Identify which raw tables are *broken* — bad enough that they need attention
before any dbt staging/transformation work touches them.

This notebook reads the `audit.dq_reports` table (already computed by the pipeline for
every table in `raw`) to answer:

- Which tables failed their quality checks outright?
- Which tables have the most duplicate rows?
- Which tables have the most columns affected by nulls?
- Do any tables have more than one report (re-run), and if so, which run should we trust?

This notebook does **not** query `raw.*` tables directly — that's what 04 is for.
Here we only read the pre-computed quality metadata.

Output of this notebook: a **shortlist of tables that need cleaning attention**, with reasons, saved at the bottom.

In [1]:
from _bootstrap import project_root
from src.audit.run_management import get_latest_run_id, list_run_ids

import polars as pl

# Show all columns and full-width string values in every cell below —
# otherwise polars truncates wide tables with "…" and cuts long strings like file_path/error
pl.Config.set_tbl_cols(-1)          # show all columns, no collapsing
pl.Config.set_tbl_width_chars(200)  # widen the rendered table
pl.Config.set_fmt_str_lengths(120)  # don't truncate long strings (e.g. file_path, error messages)
pl.Config.set_tbl_rows(50)          # show more rows before truncating vertically

# Connect to the project's DuckDB instance
from src.database.connection import get_duckdb_conn

conn = get_duckdb_conn(True)
print("Connected")

Connected


## 1. Load `audit.dq_reports`

Pull the full table into polars. We know from 01 that this has one row per
(table_name, run_id) — most tables have exactly one report, a handful have two
(re-runs after a pipeline fix).

In [2]:
dq_df = conn.execute("""
    SELECT run_id, layer, table_name, generated_at, report_version,
           status, total_rows, duplicate_rows, columns_with_nulls, failure_reasons
    FROM audit.dq_reports
""").pl()

print(f"Total dq_reports rows: {dq_df.height}")
print(f"Distinct tables covered: {dq_df['table_name'].n_unique()}")
dq_df.head(10)

Total dq_reports rows: 738
Distinct tables covered: 369


run_id,layer,table_name,generated_at,report_version,status,total_rows,duplicate_rows,columns_with_nulls,failure_reasons
str,str,str,str,i32,str,i64,i64,i64,str
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures""","""2026-07-07T18:08:08""",1,"""PASS""",7789,0,1,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_archive""","""2026-07-07T18:08:08""",1,"""PASS""",3094,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_archive_areacodes""","""2026-07-07T18:08:08""",1,"""PASS""",121,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_archive_elements""","""2026-07-07T18:08:08""",1,"""PASS""",2,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_archive_flags""","""2026-07-07T18:08:08""",1,"""PASS""",1,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_archive_itemcodes""","""2026-07-07T18:08:08""",1,"""PASS""",1,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_areacodes""","""2026-07-07T18:08:08""",1,"""PASS""",163,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_costcategorys""","""2026-07-07T18:08:08""",1,"""PASS""",1,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_flags""","""2026-07-07T18:08:08""",1,"""PASS""",5,0,0,"""[]"""


## 2. Resolve to latest run per table

Use `get_latest_run_id()` (from `src.audit.run_management`) to identify the most
recent `raw` run, then take that run's report for each table. Since we already know
some tables were profiled in an earlier run and never re-run, confirm coverage and
backfill any table missing from the latest run with its own most recent report —
so nothing silently drops out.

In [3]:
latest_run_id = get_latest_run_id(conn, "audit.dq_reports", layer="raw")
print(f"Latest dq_reports run_id: {latest_run_id}")

latest_dq = dq_df.filter(pl.col("run_id") == latest_run_id)

missing_from_latest = set(dq_df["table_name"]) - set(latest_dq["table_name"])
if missing_from_latest:
    print(f"{len(missing_from_latest)} tables missing from latest run — backfilling from their own latest report")
    backfill = (
        dq_df
        .filter(pl.col("table_name").is_in(missing_from_latest))
        .sort("generated_at", descending=True)
        .unique(subset=["table_name"], keep="first")
    )
    latest_dq = pl.concat([latest_dq, backfill])
else:
    print("All tables are covered by the latest run — no backfill needed.")

print(f"Final table count: {latest_dq.height}")
assert latest_dq.height == dq_df["table_name"].n_unique(), "Should cover every table exactly once"
print("OK — one row per table, full coverage confirmed.")

Latest dq_reports run_id: raw_2026-07-07T18:57:29
All tables are covered by the latest run — no backfill needed.
Final table count: 369
OK — one row per table, full coverage confirmed.


### Which tables were re-run, and why?

Worth a quick look before moving on — if the two runs disagree meaningfully
(e.g. status flipped from fail to pass), that's useful context. If they're
identical bar timestamp, it's safe to ignore and just trust latest.

In [4]:
reran_tables = (
    dq_df
    .group_by("table_name")
    .agg(pl.count("run_id").alias("n_reports"))
    .filter(pl.col("n_reports") > 1)
    .sort("table_name")
)

print(f"Tables with more than one dq_report: {reran_tables.height}")

# .to_list() avoids polars' is_in-with-Series deprecation warning
dq_df.filter(pl.col("table_name").is_in(reran_tables["table_name"].to_list())).sort(["table_name", "generated_at"])

Tables with more than one dq_report: 369


run_id,layer,table_name,generated_at,report_version,status,total_rows,duplicate_rows,columns_with_nulls,failure_reasons
str,str,str,str,i32,str,i64,i64,i64,str
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures""","""2026-07-07T18:08:08""",1,"""PASS""",7789,0,1,"""[]"""
"""raw_2026-07-07T18:57:29""","""raw""","""asti_expenditures""","""2026-07-07T18:56:58""",1,"""PASS""",7789,0,1,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_archive""","""2026-07-07T18:08:08""",1,"""PASS""",3094,0,0,"""[]"""
"""raw_2026-07-07T18:57:29""","""raw""","""asti_expenditures_archive""","""2026-07-07T18:56:58""",1,"""PASS""",3094,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_archive_areacodes""","""2026-07-07T18:08:08""",1,"""PASS""",121,0,0,"""[]"""
"""raw_2026-07-07T18:57:29""","""raw""","""asti_expenditures_archive_areacodes""","""2026-07-07T18:56:58""",1,"""PASS""",121,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_archive_elements""","""2026-07-07T18:08:08""",1,"""PASS""",2,0,0,"""[]"""
"""raw_2026-07-07T18:57:29""","""raw""","""asti_expenditures_archive_elements""","""2026-07-07T18:56:58""",1,"""PASS""",2,0,0,"""[]"""
"""raw_2026-07-07T18:08:39""","""raw""","""asti_expenditures_archive_flags""","""2026-07-07T18:08:08""",1,"""PASS""",1,0,0,"""[]"""


## 3. Outright failures

Tables where `status != 'pass'` or `failure_reasons` is non-empty. These are the
highest priority — the pipeline itself flagged something wrong.

In [5]:
# NOTE: status is stored uppercase (e.g. "PASS"), confirmed from actual output —
# comparing against lowercase "pass" would silently flag every table as a failure.
failures = (
    latest_dq
    .filter(
        (pl.col("status").str.to_uppercase() != "PASS") | (pl.col("failure_reasons") != "[]")
    )
    .sort("table_name")
)

print(f"Tables with a non-pass status or non-empty failure_reasons: {failures.height}")
failures

Tables with a non-pass status or non-empty failure_reasons: 3


run_id,layer,table_name,generated_at,report_version,status,total_rows,duplicate_rows,columns_with_nulls,failure_reasons
str,str,str,str,i32,str,i64,i64,i64,str
"""raw_2026-07-07T18:57:29""","""raw""","""foodbalancesheets_areacodes""","""2026-07-07T18:57:07""",1,"""FAIL""",426,213,0,"""[{""reason"": ""duplicate_rows"", ""detail"": ""213 full-row duplicates found (426 total -> 213 distinct).""}]"""
"""raw_2026-07-07T18:57:29""","""raw""","""foodbalancesheetshistoric_areacodes""","""2026-07-07T18:57:09""",1,"""FAIL""",434,217,0,"""[{""reason"": ""duplicate_rows"", ""detail"": ""217 full-row duplicates found (434 total -> 217 distinct).""}]"""
"""raw_2026-07-07T18:57:29""","""raw""","""forestry_trade_flows_areacodes""","""2026-07-07T18:57:10""",1,"""FAIL""",423,207,0,"""[{""reason"": ""duplicate_rows"", ""detail"": ""207 full-row duplicates found (423 total -> 216 distinct).""}]"""


## 4. Duplicate rows

Ranked by `duplicate_rows` descending. High duplicate counts usually mean either a raw
load-time issue (double-ingested file) or a grain misunderstanding (the table is wider
grain than the natural key suggests).

In [6]:
dupe_ranked = (
    latest_dq
    .filter(pl.col("duplicate_rows") > 0)
    .select(["table_name", "total_rows", "duplicate_rows"])
    .with_columns(
        (pl.col("duplicate_rows") / pl.col("total_rows") * 100).round(2).alias("duplicate_pct")
    )
    .sort("duplicate_rows", descending=True)
)

print(f"Tables with any duplicate rows: {dupe_ranked.height}")
dupe_ranked.head(30)

Tables with any duplicate rows: 3


table_name,total_rows,duplicate_rows,duplicate_pct
str,i64,i64,f64
"""foodbalancesheetshistoric_areacodes""",434,217,50.0
"""foodbalancesheets_areacodes""",426,213,50.0
"""forestry_trade_flows_areacodes""",423,207,48.94


### Is this systemic to `_areacodes` tables, or isolated to these 3?

All 3 failures above are `_areacodes` lookup tables with ~50% exact duplicate rows —
too consistent a pattern to be a coincidence. Before treating each as a one-off,
check whether every `_areacodes` table in `raw` has this problem, or just these 3.
If it's every one of them, the fix belongs in the loader (dedupe on ingest), not in
three separate staging models.

In [7]:
areacodes_check = (
    latest_dq
    .filter(pl.col("table_name").str.contains("areacodes"))
    .select(["table_name", "status", "total_rows", "duplicate_rows"])
    .with_columns(
        (pl.col("duplicate_rows") / pl.col("total_rows") * 100).round(2).alias("duplicate_pct")
    )
    .sort("duplicate_pct", descending=True)
)

n_total = areacodes_check.height
n_affected = areacodes_check.filter(pl.col("duplicate_rows") > 0).height

print(f"_areacodes tables total: {n_total}")
print(f"_areacodes tables with any duplicate rows: {n_affected}")
areacodes_check

_areacodes tables total: 64
_areacodes tables with any duplicate rows: 3


table_name,status,total_rows,duplicate_rows,duplicate_pct
str,str,i64,i64,f64
"""foodbalancesheets_areacodes""","""FAIL""",426,213,50.0
"""foodbalancesheetshistoric_areacodes""","""FAIL""",434,217,50.0
"""forestry_trade_flows_areacodes""","""FAIL""",423,207,48.94
"""asti_expenditures_archive_areacodes""","""PASS""",121,0,0.0
"""asti_expenditures_areacodes""","""PASS""",163,0,0.0
"""asti_researchers_archive_areacodes""","""PASS""",121,0,0.0
"""asti_researchers_areacodes""","""PASS""",168,0,0.0
"""climate_change_emissions_indicators_areacodes""","""PASS""",279,0,0.0
"""commoditybalances_non_food_2010_areacodes""","""PASS""",213,0,0.0


## 5. Columns affected by nulls

Ranked by `columns_with_nulls` descending. This is a *count of columns*, not rows —
a high value here means the table's schema is largely sparse, which matters for
deciding what's safe to make `NOT NULL` downstream.

In [8]:
nulls_ranked = (
    latest_dq
    .filter(pl.col("columns_with_nulls") > 0)
    .select(["table_name", "total_rows", "columns_with_nulls"])
    .sort("columns_with_nulls", descending=True)
)

print(f"Tables with at least one column containing nulls: {nulls_ranked.height}")
nulls_ranked.head(30)

Tables with at least one column containing nulls: 57


table_name,total_rows,columns_with_nulls
str,i64,i64
"""trade_matrix""",6640547,78
"""emdat""",27681,29
"""commodity_prices""",798,26
"""holidays""",88537,4
"""commoditybalances_non_food_2010""",127558,2
"""emissions_crops""",766730,2
"""employment_indicators_agriculture""",256389,2
"""food_security_data""",279470,2
"""production_crops_livestock""",4209110,2


## 6. Build the shortlist

Combine the three signals above into a single prioritized list: any table that shows up
in failures, top duplicates, or top nulls gets flagged with the specific reason(s).
This is the actionable output of the notebook — not all 369 tables, just the ones that
need a decision before staging.

In [9]:
TOP_N_DUPES = 20   # adjust threshold after eyeballing section 4
TOP_N_NULLS = 20   # adjust threshold after eyeballing section 5

flagged_failure = set(failures["table_name"].to_list())
flagged_dupes = set(dupe_ranked.head(TOP_N_DUPES)["table_name"].to_list())
flagged_nulls = set(nulls_ranked.head(TOP_N_NULLS)["table_name"].to_list())

all_flagged = flagged_failure | flagged_dupes | flagged_nulls

def reasons_for(table_name: str) -> str:
    reasons = []
    if table_name in flagged_failure:
        reasons.append("failed_check")
    if table_name in flagged_dupes:
        reasons.append("high_duplicates")
    if table_name in flagged_nulls:
        reasons.append("high_null_columns")
    return ", ".join(reasons)

shortlist = (
    latest_dq
    .filter(pl.col("table_name").is_in(all_flagged))
    .select(["table_name", "status", "total_rows", "duplicate_rows", "columns_with_nulls"])
    .with_columns(
        pl.col("table_name").map_elements(reasons_for, return_dtype=pl.Utf8).alias("flagged_for")
    )
    .sort("table_name")
)

print(f"Tables flagged for follow-up: {shortlist.height} / {latest_dq.height}")
shortlist

Tables flagged for follow-up: 23 / 369


table_name,status,total_rows,duplicate_rows,columns_with_nulls,flagged_for
str,str,i64,i64,i64,str
"""asti_expenditures""","""PASS""",7789,0,1,"""high_null_columns"""
"""asti_researchers""","""PASS""",3800,0,1,"""high_null_columns"""
"""commodity_prices""","""PASS""",798,0,26,"""high_null_columns"""
"""commoditybalances_non_food_2010""","""PASS""",127558,0,2,"""high_null_columns"""
"""consumerpriceindices""","""PASS""",248394,0,1,"""high_null_columns"""
"""cost_affordability_healthy_diet_coahd""","""PASS""",11672,0,1,"""high_null_columns"""
"""development_assistance_to_agriculture""","""PASS""",13020275,0,1,"""high_null_columns"""
"""emdat""","""PASS""",27681,0,29,"""high_null_columns"""
"""emissions_crops""","""PASS""",766730,0,2,"""high_null_columns"""


### Notes — Data Quality

- *(fill in after reviewing the shortlist above)*
- Any re-run tables (section 2) where status flipped — worth understanding what changed upstream?
- Are the high-duplicate tables (section 4) genuine data problems, or expected given the table's grain
  (e.g. a long/normalized FAOSTAT table with repeated dimension values isn't the same as duplicate *rows*)?
- **Resolved:** duplicate rows are isolated, not systemic — only 3 of 64 `_areacodes` tables are affected
  (`foodbalancesheets_areacodes`, `foodbalancesheetshistoric_areacodes`, `forestry_trade_flows_areacodes`).
  The other 61 are clean, so this is not a loader-level bug — it points to those 3 specific source files
  shipping duplicated rows. Fix in 05: `distinct()` on these 3 tables in staging, not a pipeline-wide change.
- Are the high-null-column tables (section 5) lookup/dimension tables where sparsity is expected,
  or fact tables where it's a real gap?
- Carry the `shortlist` table_names forward into **04 — Raw Data Exploration** for a closer look at
  actual values, and into **05 — Data Cleaning Plan** for concrete staging decisions.

In [10]:
# Close the connection
conn.close()
print("Connection closed")

Connection closed
